# Import dependencies

In [1]:
import pandas as pd

### Load raw MT4/MT5 data

In [ ]:
file_path = "./xauusd_15m.csv"

df = pd.read_csv(
    file_path,
    sep="\t",
    engine="python"
)

### 2. Clean column names

In [3]:
df.columns = [c.strip("<>").lower() for c in df.columns]

# rename for clarity
df.rename(columns={
    "date": "date",
    "time": "time",
    "open": "open",
    "high": "high",
    "low": "low",
    "close": "close",
    "tickvol": "tickvol",
    "spread": "spread"
}, inplace=True)

### 3. Combine date + time

In [4]:
df["datetime"] = pd.to_datetime(
    df["date"] + " " + df["time"],
    format="%Y.%m.%d %H:%M:%S"
)

df = df.sort_values("datetime").reset_index(drop=True)

### 4. Create LLM-friendly features

In [5]:
df["range"] = df["high"] - df["low"]
df["body"] = abs(df["close"] - df["open"])

df["direction"] = df.apply(
    lambda x: "BULL" if x["close"] > x["open"]
    else "BEAR" if x["close"] < x["open"]
    else "DOJI",
    axis=1
)

### 5. Select final columns

In [6]:
final_df = df[
    [
        "datetime",
        "open",
        "high",
        "low",
        "close",
        "range",
        "body",
        "direction",
        "tickvol",
        "spread"
    ]
]

### 6. Round numbers (important for LLM clarity)

In [7]:
price_cols = ["open", "high", "low", "close", "range", "body"]
final_df[price_cols] = final_df[price_cols].round(2)

/var/folders/kc/w9l6xj056fn5vq583fjcnqg00000gn/T/ipykernel_61734/871112631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df[price_cols] = final_df[price_cols].round(2)


### 7. Export for ChatGPT

In [8]:
final_df.to_csv("final.csv", index=False)

print("✅ Data preprocessed and ready for LLM analysis")

✅ Data preprocessed and ready for LLM analysis
